# 📊 Superstore Sales Analysis (2015–2018)

**Author:** Rohan Bhosale  
**Dataset:** [Superstore Sales Dataset — Kaggle](https://www.kaggle.com/datasets/rohitsahoo/sales-forecasting)  
**Tech Stack:** Python · Pandas · NumPy · Matplotlib · Seaborn

---

### Objectives
- Explore 4 years of US retail sales data (9,800 orders across 2015–2018)
- Identify top-performing categories, regions, and customer segments
- Analyse year-over-year revenue growth and monthly seasonality
- Surface actionable business insights from the data

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 130

print("Libraries loaded ✅")

## 2. Load & Inspect Data

In [ ]:
df = pd.read_csv("sales_data.csv")
df["Order Date"] = pd.to_datetime(df["Order Date"], dayfirst=True)
df["Ship Date"]  = pd.to_datetime(df["Ship Date"],  dayfirst=True)

print("=" * 55)
print("📦  DATASET OVERVIEW")
print("=" * 55)
print(f"  Rows        : {df.shape[0]:,}")
print(f"  Columns     : {df.shape[1]}")
print(f"  Date range  : {df['Order Date'].min().date()} → {df['Order Date'].max().date()}")
print(f"  Years       : {sorted(df['Order Date'].dt.year.unique().tolist())}")
print(f"  Categories  : {df['Category'].unique().tolist()}")
print(f"  Regions     : {df['Region'].unique().tolist()}")
print(f"  Segments    : {df['Segment'].unique().tolist()}")
print(f"\n  Missing values:")
missing = df.isnull().sum()
print(missing[missing > 0].to_string() if missing.any() else "  None ✅")

In [ ]:
df.head()

## 3. Data Cleaning & Feature Engineering

In [ ]:
before = len(df)

# Drop rows with missing Sales (primary metric)
df = df.dropna(subset=["Sales"])

# Fill missing Postal Code with 0 (non-critical for analysis)
df["Postal Code"] = df["Postal Code"].fillna(0).astype(int)

# Feature engineering
df["Year"]          = df["Order Date"].dt.year
df["Month"]         = df["Order Date"].dt.month
df["Month Name"]    = df["Order Date"].dt.strftime("%b")
df["Quarter"]       = df["Order Date"].dt.to_period("Q").astype(str)
df["Days to Ship"]  = (df["Ship Date"] - df["Order Date"]).dt.days

print(f"🧹  Rows before: {before:,} → after: {len(df):,} (dropped {before - len(df)} rows)")
print(f"✅  New features added: Year, Month, Quarter, Days to Ship")
print(f"\n  Days to Ship — mean: {df['Days to Ship'].mean():.1f} days | max: {df['Days to Ship'].max()} days")

## 4. Key Business Metrics

In [ ]:
total_sales   = df["Sales"].sum()
avg_order     = df["Sales"].mean()
total_orders  = df["Order ID"].nunique()
top_category  = df.groupby("Category")["Sales"].sum().idxmax()
top_state     = df.groupby("State")["Sales"].sum().idxmax()
top_segment   = df.groupby("Segment")["Sales"].sum().idxmax()

print("=" * 55)
print("📊  KEY METRICS (All Years Combined)")
print("=" * 55)
print(f"  Total Revenue      : ${total_sales:,.2f}")
print(f"  Unique Orders      : {total_orders:,}")
print(f"  Avg Order Value    : ${avg_order:,.2f}")
print(f"  Top Category       : {top_category}")
print(f"  Top State          : {top_state}")
print(f"  Top Segment        : {top_segment}")

print("\n  Revenue by Region:")
for r, v in df.groupby("Region")["Sales"].sum().sort_values(ascending=False).items():
    print(f"    {r:<10} ${v:,.0f}")

print("\n  Revenue by Category:")
for c, v in df.groupby("Category")["Sales"].sum().sort_values(ascending=False).items():
    print(f"    {c:<20} ${v:,.0f}")

print("\n  Revenue by Year:")
for y, v in df.groupby("Year")["Sales"].sum().items():
    print(f"    {y}    ${v:,.0f}")

## 5. Visualisations

### 5.1 Monthly Revenue Trend (All Years Overlaid)

In [ ]:
month_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

monthly_by_year = (
    df.groupby(["Year", "Month", "Month Name"])["Sales"]
    .sum()
    .reset_index()
    .sort_values(["Year", "Month"])
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: all years overlaid
palette = sns.color_palette("tab10", 4)
for i, (year, grp) in enumerate(monthly_by_year.groupby("Year")):
    axes[0].plot(
        grp["Month"], grp["Sales"],
        marker="o", linewidth=2, label=str(year), color=palette[i]
    )
axes[0].set_xticks(range(1, 13))
axes[0].set_xticklabels(month_order)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}k"))
axes[0].set_title("Monthly Revenue by Year", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Month")
axes[0].set_ylabel("Revenue")
axes[0].legend(title="Year")

# Right: Year-over-Year total
yoy = df.groupby("Year")["Sales"].sum().reset_index()
bars = axes[1].bar(yoy["Year"].astype(str), yoy["Sales"],
                   color=palette, edgecolor="white", linewidth=0.5)
for bar, val in zip(bars, yoy["Sales"]):
    axes[1].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 2000,
                 f"${val/1e3:.0f}k", ha="center", fontsize=10, fontweight="bold")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}k"))
axes[1].set_title("Year-over-Year Revenue", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Total Revenue")

plt.suptitle("Revenue Trends — Superstore 2015–2018", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("monthly_revenue.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: monthly_revenue.png")

### 5.2 Category & Sub-Category Breakdown

In [ ]:
cat_rev    = df.groupby("Category")["Sales"].sum().sort_values()
subcat_rev = df.groupby("Sub-Category")["Sales"].sum().sort_values(ascending=False).head(10)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors3 = sns.color_palette("muted", 3)

# Pie chart
axes[0].pie(
    cat_rev.values, labels=cat_rev.index,
    autopct="%1.1f%%", colors=colors3,
    startangle=140, pctdistance=0.82
)
axes[0].set_title("Revenue Share by Category", fontweight="bold")

# Category bar
axes[1].barh(cat_rev.index, cat_rev.values, color=colors3)
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}k"))
axes[1].set_title("Total Revenue by Category", fontweight="bold")
axes[1].set_xlabel("Revenue")

# Top 10 sub-categories
subcat_colors = sns.color_palette("coolwarm", len(subcat_rev))
axes[2].barh(subcat_rev.index[::-1], subcat_rev.values[::-1], color=subcat_colors)
axes[2].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}k"))
axes[2].set_title("Top 10 Sub-Categories by Revenue", fontweight="bold")
axes[2].set_xlabel("Revenue")

plt.suptitle("Category Performance — Superstore 2015–2018", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("category_breakdown.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: category_breakdown.png")

### 5.3 Region × Category Revenue Heatmap

In [ ]:
pivot = df.pivot_table(
    index="Region", columns="Category",
    values="Sales", aggfunc="sum"
)

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(
    pivot, annot=True, fmt=",.0f",
    cmap="YlOrRd", linewidths=0.5, ax=ax,
    cbar_kws={"label": "Revenue ($)"}
)
ax.set_title("Revenue Heatmap — Region × Category", fontsize=13, fontweight="bold")
ax.set_xlabel("Category")
ax.set_ylabel("Region")
plt.tight_layout()
plt.savefig("region_category_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: region_category_heatmap.png")

### 5.4 Customer Segment Analysis

In [ ]:
seg_rev     = df.groupby("Segment")["Sales"].sum().sort_values(ascending=False)
seg_orders  = df.groupby("Segment")["Order ID"].nunique().sort_values(ascending=False)
seg_avg     = (seg_rev / seg_orders).sort_values(ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
seg_colors = sns.color_palette("Set2", 3)

# Total revenue by segment
bars = axes[0].bar(seg_rev.index, seg_rev.values, color=seg_colors, edgecolor="white")
for bar, val in zip(bars, seg_rev.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 1000,
                 f"${val/1e3:.0f}k", ha="center", fontsize=10, fontweight="bold")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}k"))
axes[0].set_title("Total Revenue by Segment", fontweight="bold")
axes[0].set_ylabel("Revenue")

# Order count by segment
bars2 = axes[1].bar(seg_orders.index, seg_orders.values, color=seg_colors, edgecolor="white")
for bar, val in zip(bars2, seg_orders.values):
    axes[1].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 5,
                 str(val), ha="center", fontsize=10, fontweight="bold")
axes[1].set_title("Order Count by Segment", fontweight="bold")
axes[1].set_ylabel("Number of Orders")

# Average order value by segment
bars3 = axes[2].bar(seg_avg.index, seg_avg.values, color=seg_colors, edgecolor="white")
for bar, val in zip(bars3, seg_avg.values):
    axes[2].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.5,
                 f"${val:.0f}", ha="center", fontsize=10, fontweight="bold")
axes[2].set_title("Avg Order Value by Segment", fontweight="bold")
axes[2].set_ylabel("Avg Revenue per Order ($)")

plt.suptitle("Customer Segment Analysis — Superstore 2015–2018",
             fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("segment_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: segment_analysis.png")

### 5.5 Top 10 States by Revenue

In [ ]:
top_states = (
    df.groupby("State")["Sales"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(11, 5))
bar_colors = sns.color_palette("Blues_r", len(top_states))
bars = ax.barh(top_states.index[::-1], top_states.values[::-1], color=bar_colors[::-1])

for bar, val in zip(bars, top_states.values[::-1]):
    ax.text(bar.get_width() + 500, bar.get_y() + bar.get_height() / 2,
            f"${val/1e3:.0f}k", va="center", fontsize=10, fontweight="bold")

ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}k"))
ax.set_title("Top 10 States by Revenue — Superstore 2015–2018",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Total Revenue")
plt.tight_layout()
plt.savefig("top_states.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: top_states.png")

## 6. Key Insights

In [ ]:
yoy_vals  = df.groupby("Year")["Sales"].sum()
yoy_growth = yoy_vals.pct_change().dropna() * 100
top10_sub  = df.groupby("Sub-Category")["Sales"].sum().sort_values(ascending=False).head(3)
q4_share   = df[df["Quarter"].str.contains("Q4")]["Sales"].sum() / df["Sales"].sum() * 100

print("=" * 55)
print("💡  KEY INSIGHTS")
print("=" * 55)

print("\n📈  Year-over-Year Revenue Growth:")
for y, g in yoy_growth.items():
    arrow = "▲" if g > 0 else "▼"
    print(f"    {y}: {arrow} {g:+.1f}%")

print(f"\n🏆  Top 3 Sub-Categories by Revenue:")
for i, (sub, val) in enumerate(top10_sub.items(), 1):
    print(f"    {i}. {sub:<15} ${val:,.0f}")

print(f"\n📅  Q4 Revenue Share: {q4_share:.1f}% of annual total")
print(f"    → Strong end-of-year seasonality visible across all years")

print(f"\n🌎  West region leads in total revenue")
print(f"    California alone accounts for the highest state-level revenue")

print(f"\n👥  Consumer segment drives the most orders")
print(f"    Corporate segment has a higher average order value")

## 7. Summary

| Insight | Finding |
|---|---|
| **Best Category** | Technology leads in total revenue |
| **Best Region** | West region dominates sales |
| **Top State** | California |
| **Seasonality** | Q4 consistently highest across all years |
| **Segment** | Consumer = most orders; Corporate = higher AOV |
| **Growth** | Revenue grew year-on-year from 2015 to 2018 |

---

### Possible Extensions
- Add shipping mode analysis (Standard vs First Class vs Same Day)
- Build a sales forecasting model using ARIMA or Prophet
- Create an interactive Streamlit dashboard with regional filters
- Customer cohort analysis by first purchase year